In [1]:
import numpy as np
from tqdm import tqdm
import torch
import torchvision
import torchvision.transforms as transforms

from multidim_bpe import *

%run ./multidim_bpe.py

c:\Users\weiwe\anaconda3\envs\bpe\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
batch_size = 1

random_seed = 1
torch.backends.cudnn.enabled = False
torch.manual_seed(random_seed)

transform = transforms.Compose([
    transforms.ToTensor(),
    lambda x: (x * 255).to(dtype=torch.int32)
])

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=False)

In [3]:
dim = (2, 2)
tokenizer = Tokenizer()
tokenized_tuples = []
labels = []

vocab_len = tokenizer.get_vocab_len()
progress = tqdm(train_loader, desc=f"vocab size [{vocab_len}]");
for image, label in progress:
    ttl = reshape_to_tuples(image, dim)
    tokenized = tokenizer.train_encode(ttl, 10000, min_freq=4)
    tokenized_tuples.append(tokenized)
    labels.append(label.item())

    vocab_len = tokenizer.get_vocab_len()
    progress.set_description(f"vocab size [{vocab_len}]")

vocab size [0]:   0%|          | 0/60000 [00:00<?, ?it/s]

vocab size [10000]: 100%|██████████| 60000/60000 [07:15<00:00, 137.91it/s]


In [4]:
tokenizer.get_vocab()

defaultdict(str,
            {'0': ((0, 0, 0, 0), (0, 0, 0, 0)),
             '1': ('0', '0'),
             '2': ('1', '1'),
             '3': ((253, 0, 253, 0), (253, 0, 253, 0)),
             '4': ('1', '0'),
             '5': ('0', (0, 0, 0, 0)),
             '6': ('2', (0, 0, 0, 0)),
             '7': ((0, 0, 251, 0), (0, 0, 251, 0)),
             '8': ((0, 0, 0, 253), (0, 0, 0, 252)),
             '9': ('2', '0'),
             '10': ((0, 252, 0, 0), (0, 252, 0, 0)),
             '11': ((0, 0, 252, 0), (0, 0, 252, 0)),
             '12': ((252, 0, 0, 0), (252, 0, 0, 0)),
             '13': ((0, 0, 0, 252), (0, 0, 0, 252)),
             '14': ('10', (0, 60, 0, 0)),
             '15': ('1', '4'),
             '16': ((0, 0, 0, 35), '4'),
             '17': ('9', (0, 0, 0, 0)),
             '18': ('6', (20, 0, 0, 0)),
             '19': ('18', (254, 0, 0, 0)),
             '20': ((0, 254, 0, 0), (0, 254, 0, 0)),
             '21': ((0, 0, 0, 254), (0, 0, 0, 254)),
             '22': ((

In [5]:
tokenizer.json_save(f'./bpe_model/vocab_{tokenizer.get_vocab_len()}.json')